In [ ]:
import json
import asyncio
import pandas as pd
from datetime import datetime, timedelta, timezone
import re
import os
from typing import List, Dict, Any
from collections import Counter

from telethon import TelegramClient
from telethon.tl.functions.channels import GetFullChannelRequest

In [ ]:
BASE_DIR = os.getcwd()
CONFIG_PATH = os.path.join(BASE_DIR, 'config.json')
DATA_DIR = os.path.join(BASE_DIR, 'data/telegram')

# Создаем папку для данных, если ее нет
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
# Загрузка конфигурации
try:
    with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
        config = json.load(f)
except FileNotFoundError:
    print(f"Создайте файл {CONFIG_PATH} с настройками:")
    print('''
{
  "telegram_api_id": "YOUR_API_ID",
  "telegram_api_hash": "YOUR_API_HASH",
  "telegram_phone": "+79001234567"
}
    ''')
    raise

# Telegram API конфигурация
API_ID = config['telegram_api_id']
API_HASH = config['telegram_api_hash']
PHONE = config['telegram_phone']

In [ ]:
class TelegramStatsCollector:
    def __init__(self, api_id: int, api_hash: str, phone: str):
        self.api_id = api_id
        self.api_hash = api_hash
        self.phone = phone
        self.client = None
    
    async def connect(self):
        """Подключение к Telegram API"""
        self.client = TelegramClient('session', self.api_id, self.api_hash)
        await self.client.start(phone=self.phone)
        print("✅ Подключение к Telegram успешно")
    
    async def get_channel_info(self, channel_username: str) -> Dict[str, Any]:
        """Получение основной информации о канале"""
        try:
            entity = await self.client.get_entity(channel_username)
            full_channel = await self.client(GetFullChannelRequest(entity))
            
            return {
                'id': entity.id,
                'title': entity.title,
                'username': entity.username,
                'participants_count': full_channel.full_chat.participants_count,
                'description': full_channel.full_chat.about,
                'created_date': entity.date,
                'verified': entity.verified if hasattr(entity, 'verified') else False,
                'restricted': entity.restricted if hasattr(entity, 'restricted') else False
            }
        except Exception as e:
            print(f"❌ Ошибка получения информации о канале: {e}")
            return {}
    
    async def get_messages(self, channel_username: str, limit: int = 1000, days_back: int = 30) -> List[Dict]:
        """Получение сообщений канала за указанный период"""
        messages = []
        
        try:
            entity = await self.client.get_entity(channel_username)
            
            # Используем простой подход без фильтра по дате
            message_count = 0
            async for message in self.client.iter_messages(entity, limit=limit):
                # Проверяем возраст сообщения после получения
                now = datetime.now(timezone.utc)
                message_age_days = (now - message.date).days
                
                # Если сообщение старше указанного периода, останавливаемся
                if message_age_days > days_back:
                    break
                    
                # Обработка grouped_id
                grouped_id = None
                is_album_item = False
                if hasattr(message, 'grouped_id') and message.grouped_id is not None:
                    grouped_id = int(message.grouped_id)  # Конвертируем в int
                    is_album_item = True
                
                msg_data = {
                    'id': message.id,
                    'date': message.date,
                    'text': message.text or '',
                    'views': message.views or 0,
                    'forwards': message.forwards or 0,
                    'replies': getattr(message.replies, 'replies', 0) if message.replies else 0,
                    'reactions': self._extract_reactions(message),
                    'media_type': self._get_media_type(message),
                    'grouped_id': grouped_id,
                    'is_album_item': is_album_item,
                    'message_length': len(message.text) if message.text else 0,
                    'has_links': bool(re.search(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', message.text or '')),
                    'mentions_count': len(re.findall(r'@\w+', message.text or '')),
                    'hashtags_count': len(re.findall(r'#\w+', message.text or ''))
                }
                messages.append(msg_data)
                message_count += 1
                
                # Показываем прогресс каждые 10 сообщений
                if message_count % 10 == 0:
                    print(f"📊 Собрано {message_count} сообщений...")
            
            print(f"✅ Собрано {len(messages)} сообщений за последние {days_back} дней")
            
            if len(messages) > 0:
                latest = messages[0]['date']
                oldest = messages[-1]['date'] if len(messages) > 1 else latest
                print(f"📅 Период: {oldest} — {latest}")
            
            return messages
            
        except Exception as e:
            print(f"❌ Ошибка получения сообщений: {e}")
            import traceback
            traceback.print_exc()
            return []
    
    def _extract_reactions(self, message) -> Dict[str, int]:
        """Извлечение реакций из сообщения"""
        reactions = {}
        if hasattr(message, 'reactions') and message.reactions:
            for reaction in message.reactions.results:
                emoji = reaction.reaction.emoticon if hasattr(reaction.reaction, 'emoticon') else str(reaction.reaction)
                reactions[emoji] = reaction.count
        return reactions
    
    def _get_media_type(self, message) -> str:
        """Определение типа медиа в сообщении"""
        if not message.media:
            return 'text'
        
        media_type = type(message.media).__name__
        
        mapping = {
            'MessageMediaPhoto': 'photo',
            'MessageMediaDocument': 'document',
            'MessageMediaWebPage': 'webpage',
            'MessageMediaGeo': 'location',
            'MessageMediaContact': 'contact',
            'MessageMediaPoll': 'poll'
        }
        
        return mapping.get(media_type, 'other')
    
    async def disconnect(self):
        """Отключение от Telegram API"""
        if self.client:
            await self.client.disconnect()
            print("✅ Отключение от Telegram")

In [ ]:
class TelegramStatsAnalyzer:
    def __init__(self, messages_data: List[Dict]):
        self.df = pd.DataFrame(messages_data)
        if not self.df.empty:
            self.df['date'] = pd.to_datetime(self.df['date'])
            self.df['hour'] = self.df['date'].dt.hour
            self.df['day_of_week'] = self.df['date'].dt.day_name()
            self.df['date_only'] = self.df['date'].dt.date
            
            # Помечаем основные посты (не медиа в альбомах)
            self.df['is_main_post'] = self._mark_main_posts()
    
    def _mark_main_posts(self) -> pd.Series:
        """Определяет основные посты (не медиа файлы в альбомах)
        
        Логика:
        - Если сообщение НЕ в альбоме (is_album_item=False) → это основной пост
        - Если сообщение В альбоме (is_album_item=True):
          - Если есть текст (message_length > 0) → это основной пост
          - Если текста нет → это медиа файл
        """
        if self.df.empty:
            return pd.Series(dtype=bool)
        
        # Обычные посты (не в альбомах) - всегда основные
        is_main = ~self.df['is_album_item'].fillna(False)
        
        # Для альбомов: основной пост = сообщение с текстом
        album_items = self.df[self.df['is_album_item'] == True]
        if not album_items.empty:
            # В альбоме основной пост = тот, у которого есть текст
            has_text = album_items['message_length'] > 0
            is_main.loc[has_text.index[has_text]] = True
        
        return is_main
    
    def get_posts_only(self) -> pd.DataFrame:
        """Возвращает DataFrame только с основными постами (без медиа файлов из альбомов)"""
        if self.df.empty:
            return pd.DataFrame()
        return self.df[self.df['is_main_post'] == True].copy()
    
    def generate_basic_stats(self, posts_only: bool = True) -> Dict[str, Any]:
        """Базовая статистика канала
        
        Args:
            posts_only: Если True, считает только основные посты (без медиа файлов из альбомов)
        """
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty:
            return {}
        
        total_views = df['views'].sum()
        total_forwards = df['forwards'].sum()
        total_replies = df['replies'].sum()
        
        return {
            'total_messages': len(df),
            'total_views': total_views,
            'total_forwards': total_forwards,
            'total_replies': total_replies,
            'avg_views_per_post': df['views'].mean(),
            'avg_forwards_per_post': df['forwards'].mean(),
            'avg_replies_per_post': df['replies'].mean(),
            'engagement_rate': (total_forwards + total_replies) / total_views * 100 if total_views > 0 else 0,
            'posts_per_day': len(df) / df['date_only'].nunique(),
            'media_distribution': df['media_type'].value_counts().to_dict(),
            'avg_message_length': df['message_length'].mean(),
            'posts_with_albums': (df['is_album_item'] == True).sum(),
            'regular_posts': (df['is_album_item'] == False).sum()
        }
    
    def get_daily_stats(self, posts_only: bool = True) -> pd.DataFrame:
        """Статистика по дням"""
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty:
            return pd.DataFrame()
        
        daily_stats = df.groupby('date_only').agg({
            'id': 'count',
            'views': 'sum',
            'forwards': 'sum',
            'replies': 'sum',
            'message_length': 'mean'
        }).rename(columns={'id': 'posts_count'})
        
        return daily_stats
    
    def get_hourly_stats(self, posts_only: bool = True) -> pd.DataFrame:
        """Статистика по часам"""
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty:
            return pd.DataFrame()
            
        hourly_stats = df.groupby('hour').agg({
            'id': 'count',
            'views': 'sum',
            'forwards': 'sum',
            'replies': 'sum'
        }).rename(columns={'id': 'posts_count'})
        
        return hourly_stats
    
    def get_weekday_stats(self, posts_only: bool = True) -> pd.DataFrame:
        """Статистика по дням недели"""
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty:
            return pd.DataFrame()
            
        weekday_stats = df.groupby('day_of_week').agg({
            'id': 'count',
            'views': 'sum',
            'forwards': 'sum',
            'replies': 'sum'
        }).rename(columns={'id': 'posts_count'})
        
        return weekday_stats
    
    def get_media_stats(self, posts_only: bool = True) -> pd.DataFrame:
        """Статистика по типам медиа"""
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty:
            return pd.DataFrame()
            
        media_stats = df.groupby('media_type').agg({
            'id': 'count',
            'views': ['sum', 'mean'],
            'forwards': ['sum', 'mean'],
            'replies': ['sum', 'mean']
        })
        
        media_stats.columns = ['posts_count', 'total_views', 'avg_views', 'total_forwards', 'avg_forwards', 'total_replies', 'avg_replies']
        return media_stats
    
    def get_top_posts(self, by: str = 'views', limit: int = 10, posts_only: bool = True) -> pd.DataFrame:
        """Топ постов по указанной метрике"""
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty:
            return pd.DataFrame()
            
        if by not in df.columns:
            return pd.DataFrame()
            
        top_posts = df.nlargest(limit, by)[['id', 'date', 'text', 'views', 'forwards', 'replies', 'media_type', 'is_album_item']]
        return top_posts
    
    def get_word_frequency(self, top_n: int = 30, posts_only: bool = True) -> Dict[str, int]:
        """Частота слов в сообщениях"""
        df = self.get_posts_only() if posts_only else self.df
        
        if df.empty or df['text'].str.len().sum() == 0:
            return {}
        
        # Объединение всех текстов
        all_text = ' '.join(df['text'].dropna())
        
        # Очистка текста
        clean_text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', all_text)
        clean_text = re.sub(r'[^а-яА-Яa-zA-Z\s]', ' ', clean_text)
        
        # Подсчет слов
        words = clean_text.lower().split()
        stop_words = {'и', 'в', 'на', 'с', 'по', 'для', 'не', 'что', 'это', 'как', 'но', 'или', 'а', 
                     'также', 'от', 'до', 'за', 'при', 'из', 'к', 'о', 'об', 'же', 'ещё', 'уже',
                     'the', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by'}
        words = [word for word in words if len(word) > 2 and word not in stop_words]
        
        word_counts = Counter(words).most_common(top_n)
        
        return dict(word_counts)
    
    def _make_path(self, filename: str) -> str:
        """Создание полного пути к файлу"""
        return os.path.join(DATA_DIR, filename)
    
    def save_data(self, filename: str = None, posts_only: bool = True) -> Dict[str, str]:
        """Сохранение всех данных в DATA_DIR
        
        Args:
            filename: Имя файла (без расширения)
            posts_only: Если True, сохраняет статистику только по основным постам
        """
        if filename is None:
            filename = f"telegram_stats_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        files_created = {}
        file_count = 0
        
        # Основные данные сообщений (всегда сохраняем все)
        if not self.df.empty:
            messages_file = self._make_path(f"{filename}_messages.csv")
            self.df.to_csv(messages_file, index=False, encoding='utf-8')
            files_created['messages'] = messages_file
            file_count += 1
        
        # Выбираем DataFrame для статистики
        df_for_stats = self.get_posts_only() if posts_only else self.df
        
        if df_for_stats.empty:
            print("⚠️ Нет данных для статистики")
            return files_created
        
        # Статистика по дням
        daily_stats = self.get_daily_stats(posts_only)
        if not daily_stats.empty:
            daily_file = self._make_path(f"{filename}_daily_stats.csv")
            daily_stats.to_csv(daily_file, encoding='utf-8')
            files_created['daily'] = daily_file
            file_count += 1
        
        # Статистика по часам
        hourly_stats = self.get_hourly_stats(posts_only)
        if not hourly_stats.empty:
            hourly_file = self._make_path(f"{filename}_hourly_stats.csv")
            hourly_stats.to_csv(hourly_file, encoding='utf-8')
            files_created['hourly'] = hourly_file
            file_count += 1
        
        # Статистика по дням недели
        weekday_stats = self.get_weekday_stats(posts_only)
        if not weekday_stats.empty:
            weekday_file = self._make_path(f"{filename}_weekday_stats.csv")
            weekday_stats.to_csv(weekday_file, encoding='utf-8')
            files_created['weekday'] = weekday_file
            file_count += 1
        
        # Статистика по типам медиа
        media_stats = self.get_media_stats(posts_only)
        if not media_stats.empty:
            media_file = self._make_path(f"{filename}_media_stats.csv")
            media_stats.to_csv(media_file, encoding='utf-8')
            files_created['media'] = media_file
            file_count += 1
        
        # Топ посты (только если есть данные)
        metrics_with_data = []
        for metric in ['views', 'forwards', 'replies']:
            top_posts = self.get_top_posts(by=metric, limit=20, posts_only=posts_only)
            if not top_posts.empty:
                top_file = self._make_path(f"{filename}_top_{metric}.csv")
                top_posts.to_csv(top_file, index=False, encoding='utf-8')
                files_created[f'top_{metric}'] = top_file
                metrics_with_data.append(metric)
                file_count += 1
        
        # Частота слов
        word_freq = self.get_word_frequency(50, posts_only)
        if word_freq:
            word_file = self._make_path(f"{filename}_word_frequency.json")
            with open(word_file, 'w', encoding='utf-8') as f:
                json.dump(word_freq, f, ensure_ascii=False, indent=2)
            files_created['words'] = word_file
            file_count += 1
        
        print(f"💾 Сохранено {file_count} файлов данных")
        if metrics_with_data:
            print(f"🏆 Топ-посты созданы для метрик: {', '.join(metrics_with_data)}")
        if posts_only:
            print(f"📊 Статистика считается только по основным постам (без медиа из альбомов)")
        
        return files_created
    
    def save_report(self, channel_info: Dict, filename: str = None, posts_only: bool = True):
        """Сохранение полного отчета в DATA_DIR"""
        if filename is None:
            filename = f"telegram_stats_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        print(f"\n💾 Сохранение отчета в {DATA_DIR}...")
        
        # Сохранение данных
        files_created = self.save_data(filename, posts_only)
        
        # Общий отчет
        report = {
            'channel_info': channel_info,
            'basic_stats': self.generate_basic_stats(posts_only),
            'collection_date': datetime.now().isoformat(),
            'posts_only_mode': posts_only,
            'files_created': files_created
        }
        
        report_file = self._make_path(f"{filename}_report.json")
        with open(report_file, 'w', encoding='utf-8') as f:
            json.dump(report, f, ensure_ascii=False, indent=2, default=str)
        
        print(f"✅ Главный отчет: {os.path.basename(report_file)}")
        print(f"📊 Всего файлов создано: {len(files_created) + 1}")
        
        return report_file, files_created

In [ ]:
# Основная функция для сбора статистики (без визуализации)
async def collect_telegram_stats(channel_username: str, days_back: int = 30, limit: int = 1000):
    """Сбор полной статистики по Telegram каналу"""
    
    collector = TelegramStatsCollector(API_ID, API_HASH, PHONE)
    
    try:
        # Подключение
        await collector.connect()
        
        # Получение информации о канале
        print(f"📊 Получение информации о канале @{channel_username}...")
        channel_info = await collector.get_channel_info(channel_username)
        
        if channel_info:
            print(f"✅ Канал: {channel_info['title']}")
            print(f"📈 Подписчиков: {channel_info['participants_count']:,}")
            print(f"📅 Создан: {channel_info['created_date']}")
        
        # Получение сообщений
        print(f"\n📝 Сбор сообщений за последние {days_back} дней...")
        messages = await collector.get_messages(channel_username, limit=limit, days_back=days_back)
        
        # Анализ данных
        if messages:
            analyzer = TelegramStatsAnalyzer(messages)
            
            # Базовая статистика
            stats = analyzer.generate_basic_stats()
            print(f"\n📊 Базовая статистика:")
            print(f"📨 Всего сообщений: {stats['total_messages']:,}")
            print(f"👁 Всего просмотров: {stats['total_views']:,}")
            print(f"🔄 Всего репостов: {stats['total_forwards']:,}")
            print(f"💬 Всего ответов: {stats['total_replies']:,}")
            print(f"📈 Вовлеченность: {stats['engagement_rate']:.2f}%")
            print(f"📅 Постов в день: {stats['posts_per_day']:.1f}")
            print(f"📏 Средняя длина сообщения: {stats['avg_message_length']:.1f} символов")
            
            # Сохранение отчета в DATA_DIR
            analyzer.save_report(channel_info, f"telegram_{channel_username}")
            
            return analyzer, channel_info
        else:
            print("❌ Не удалось получить сообщения")
            return None, channel_info
    
    except Exception as e:
        print(f"❌ Ошибка: {e}")
        return None, None
    
    finally:
        await collector.disconnect()

In [ ]:
# Используем ID канала для сбора статистики
CHANNEL_USERNAME = "naturalists_notes_st" # Или ID
DAYS_BACK = 712  # Весь 2025 год
MESSAGE_LIMIT = 2000  # Увеличиваем лимит для полной статистики

print(f"📅 Собираем статистику канала {CHANNEL_USERNAME} за последние {DAYS_BACK} дней")
analyzer, channel_info = await collect_telegram_stats(CHANNEL_USERNAME, DAYS_BACK, MESSAGE_LIMIT)